In [1]:
import pandas as pd
import plotly

In [5]:
data_full = pd.read_csv("../data/raw/rusentiment_random_posts.csv")
data_preselected = pd.read_csv("../data/raw/rusentiment_preselected_posts.csv")
data_test = pd.read_csv("../data/raw/rusentiment_test.csv")
display(data_full)

,label,text
0,negative,"А попа подозревала давно,что ты с кавказа..пер..."
1,speech,З прошедшим Днем Ангела))))))))
2,skip,Два дня до отлёта с острова!!!!!!!
3,negative,"Блин, почему эта жизнь столь не справедлива (((("
4,skip,где еще встречать свой день рождения как не на...
...,...,...
21263,neutral,"Анастасия, у меня есть друг, с которым вы хоро..."
21264,neutral,Колька пошли гулять!!?
21265,neutral,Ура! Золото дают бесплатно!Напиши это в 4 комм...
21266,speech,"С Праздником, Ксюнь! \nЖенского счастья тебе!\..."


In [6]:
def show_distribution(df, name):
    dist = df['label'].value_counts().sort_index()
    percent = (dist / dist.sum() * 100).round(2)

    result = pd.DataFrame({
        "count": dist,
        "percent": percent
    })

    total = pd.DataFrame({
        "count": [dist.sum()],
        "percent": [100.0]
    }, index=["Total"])

    result = pd.concat([result, total])

    print(f"\n{name}")
    display(result)

show_distribution(data_full, "Random posts")
show_distribution(data_preselected, "Preselected posts")
show_distribution(data_test, "Test set")


Random posts


,count,percent
negative,2294,10.79
neutral,8323,39.13
positive,4635,21.79
skip,3190,15.00
speech,2826,13.29
Total,21268,100.00



Preselected posts


,count,percent
negative,1360,19.57
neutral,2977,42.83
positive,1475,21.22
skip,904,13.01
speech,234,3.37
Total,6950,100.00



Test set


,count,percent
negative,258,8.70
neutral,1420,47.86
positive,536,18.07
skip,346,11.66
speech,407,13.72
Total,2967,100.00


In [7]:
import plotly.express as px

def plot_class_distribution(df, name):
    dist = df["label"].value_counts().reset_index()
    dist.columns = ["label", "count"]

    fig = px.bar(
        dist,
        x="label",
        y="count",
        title=f"Class Distribution ({name})"
    )

    fig.show()


plot_class_distribution(data_full, "Random posts")
plot_class_distribution(data_preselected, "Preselected posts")
plot_class_distribution(data_test, "Test set")

In [8]:
import pandas as pd
import plotly.express as px

datasets = {
    "Random posts": data_full,
    "Preselected posts": data_preselected,
    "Test set": data_test
}

rows = []

for name, df in datasets.items():
    dist = df["label"].value_counts(normalize=True) * 100
    for label, percent in dist.items():
        rows.append({
            "dataset": name,
            "label": label,
            "percent": percent
        })

dist_df = pd.DataFrame(rows)

fig = px.bar(
    dist_df,
    x="label",
    y="percent",
    color="dataset",
    barmode="group",
    title="Class Distribution Across Datasets (%)"
)

fig.show()

In [9]:
for df in [data_full, data_preselected, data_test]:
    df["text_len"] = df["text"].str.len()

fig = px.histogram(
    data_test,
    x="text_len",
    nbins=50,
    title="Text Length Distribution (Test Set)"
)

fig.show()

fig = px.box(
    data_test,
    x="label",
    y="text_len",
    title="Text Length by Class"
)

fig.show()

df_all = pd.concat([
    data_full.assign(dataset="random"),
    data_preselected.assign(dataset="preselected"),
    data_test.assign(dataset="test")
])

fig = px.box(
    df_all,
    x="dataset",
    y="text_len",
    title="Text Length Across Datasets"
)

fig.show()

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

datasets = {
    "random": data_full,
    "preselected": data_preselected,
    "test": data_test
}

for name, df in datasets.items():
    df["token_len"] = df["text"].apply(
        lambda x: len(tokenizer.encode(x, add_special_tokens=True))
    )

In [11]:
df_all = pd.concat([
    data_full.assign(dataset="random"),
    data_preselected.assign(dataset="preselected"),
    data_test.assign(dataset="test")
])

fig = px.box(
    df_all,
    x="dataset",
    y="token_len",
    title="Token Length Across Datasets"
)

fig.show()

fig = px.violin(
    df_all,
    x="dataset",
    y="token_len",
    box=True,
    points=False,
    title="Token Length Distribution Across Datasets"
)

fig.show()

In [12]:
import numpy as np
import plotly.graph_objects as go

df_plot = df_all[df_all["token_len"] <= 200]

fig = go.Figure()

for name, group in df_plot.groupby("dataset"):

    values = group["token_len"].values
    
    hist, bins = np.histogram(values, bins=60, range=(0, 200), density=True)
    centers = (bins[:-1] + bins[1:]) / 2

    fig.add_trace(
        go.Scatter(
            x=centers,
            y=hist,
            mode="lines",
            name=name
        )
    )

fig.update_layout(
    title="Token Length Distribution (Density)",
    xaxis_title="Token Length",
    yaxis_title="Density",
)

fig.show()